# News2Stock Analist LoRA Finetuning

In [2]:
# runpod에 설치 되지 않는 패키지 설치
# accelerate: 멀티 gpu 환경 분산 학습 및 최적화
# trl: sft(미세 지도 조정)을 위한 Trainer 클래스 및 설정 클래스 제공
# peft: 다양한 peft 기법 지원(LoRA)
%pip install transformers datasets accelerate peft trl hf_transfer pydantic langchain-huggingface

  Using cached transformers-5.9.0-py3-none-any.whl.metadata (33 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached trl-1.4.0-py3-none-any.whl.metadata (11 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached huggingface_hub-1.15.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.5.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x

In [3]:
# 로컬
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.environ['HF_TOKEN']

ModuleNotFoundError: No module named 'dotenv'

In [4]:
# Runpod (서버 환경변수에서 가져옴)
import os
HF_TOKEN = os.environ['HF_TOKEN']

In [5]:
# 데이터셋 로드
from datasets import load_dataset

dataset = load_dataset('sim2084/naver-economy-news2stock',split='train')
print(len(dataset))
dataset

1000


Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 1000
})

In [6]:
dataset[0]

{'system': "\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,\n이유/근거 등을 분석하는 금융 분석 전문가입니다.\n\n다음 출력 지시사항을 지켜주세요.\n1. 뉴스와 종목간의 영향성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종복간의 영향성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정 영향이 예상되는 종목이 있다면, negaitive_stocks, negaitive_keywords, negaitive_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])를 작성하세요.\n",
 'user': '추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 

## 데이터 분할

In [7]:
text_ratio = 0.2

train_data=[]
text_data=[]

data_indices = list(range(len(dataset)))
test_size = int(len(dataset)*text_ratio)

train_data_indices = data_indices[test_size:]
test_data_indices = data_indices[:test_size]

# 평가/ 학습 데이터셋 형식 지정 함수
def format_data(data):
    return {
        'messages' : [
            {'role':'system','content':data['system']},
            {'role':'user','content':data['user']},
            {'role':'assistant','content':data['assistant']},
        ]
    }

train_data = [format_data(dataset[i]) for  i in train_data_indices]
test_data = [format_data(dataset[i]) for  i in test_data_indices]

print('학습셋 :', len(train_data))
print('평가셋 :', len(test_data))

학습셋 : 800
평가셋 : 200


In [6]:
# 값 하나 확인
train_data[128]

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,\n이유/근거 등을 분석하는 금융 분석 전문가입니다.\n\n다음 출력 지시사항을 지켜주세요.\n1. 뉴스와 종목간의 영향성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종복간의 영향성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정 영향이 예상되는 종목이 있다면, negaitive_stocks, negaitive_keywords, negaitive_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])를 작성하세요.\n"},
  {'role': 'user',
   'content': '에어부산 울란바토르·오사카 노선 재개\n에어부산이 김해국제공항에서 출발하는 몽골 울란바토르와 일본 오사카 노선 운항을 각각 주 2회 일정으로 코로나19 팬데믹 사태 이후 28개월 만에 재개한다고 1일 밝혔다. 부산 울란바토르 노선은 김해국제공항에서 오전 8시 25분에 출발해 현지 공항에 오전 11시 40분 도착하고 귀국편은 오후 1시에 출발해 김해공항에 오후 5시 30분 도착하는 일정으로 주 2회 운항한다. 몽골은 입국 시 코로나19 검사와 백신 접종 여부를 확인하지 않아 자유롭게 여행이 가능한 국가다. 부산 오사카 노선은 김해공항에서 오전 8시 35분에 출발해 간사이공항에 오전 10시 도착 귀국편은 간사이공항에서 낮 12시에 출발해 김해공항에 오후 1시 30분 도착하는 일정으로 주 2회

In [8]:
# dataset 객체로 다시 변환
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

In [9]:
# 값 하나 확인 -> 내용 변환 없이 타입 변환만 수행
train_data[128]

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,\n이유/근거 등을 분석하는 금융 분석 전문가입니다.\n\n다음 출력 지시사항을 지켜주세요.\n1. 뉴스와 종목간의 영향성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종복간의 영향성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정 영향이 예상되는 종목이 있다면, negaitive_stocks, negaitive_keywords, negaitive_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])를 작성하세요.\n"},
  {'role': 'user',
   'content': '에어부산 울란바토르·오사카 노선 재개\n에어부산이 김해국제공항에서 출발하는 몽골 울란바토르와 일본 오사카 노선 운항을 각각 주 2회 일정으로 코로나19 팬데믹 사태 이후 28개월 만에 재개한다고 1일 밝혔다. 부산 울란바토르 노선은 김해국제공항에서 오전 8시 25분에 출발해 현지 공항에 오전 11시 40분 도착하고 귀국편은 오후 1시에 출발해 김해공항에 오후 5시 30분 도착하는 일정으로 주 2회 운항한다. 몽골은 입국 시 코로나19 검사와 백신 접종 여부를 확인하지 않아 자유롭게 여행이 가능한 국가다. 부산 오사카 노선은 김해공항에서 오전 8시 35분에 출발해 간사이공항에 오전 10시 도착 귀국편은 간사이공항에서 낮 12시에 출발해 김해공항에 오후 1시 30분 도착하는 일정으로 주 2회

# NCSOFT/Llama-VARCO-8B-Instruct란?
https://huggingface.co/NCSOFT/Llama-VARCO-8B-Instruct

- 기본 모델: Meta의 Llama-3.1-8B 모델을 기반으로 한다.
- 개발 목적: 한국어 능력을 극대화하는 동시에 영어 구사 능력도 유지하도록 설계되었다.
- 학습 방법: 한국어와 영어 데이터셋을 활용한 지속 사전 학습(Continual Pre-training)을 거쳤으며, 이후 지도 미세 조정(SFT)과 직접 선호도 최적화(DPO)를 통해 인간의 선호도에 맞게 정렬되었다.

## SFT에서 한국어능력향상과 동시에 영어능력유지란?
일반적으로 한국어 데이터를 대량으로 추가 학습시키면 기존에 모델이 가지고 있던 영어 지식이 손상되는 파괴적 망각(Catastrophic Forgetting) 현상이 발생한다. 엔씨소프트는 이를 방지하기 위해 지속 사전 학습(Continual Pre-training)을 적용했다.

### 1. 데이터 믹스(Data Mixing) 전략
단순히 한국어 데이터만 밀어 넣는 것이 아니라, 모델이 이미 학습했던 영어 데이터와 고품질의 한국어 데이터를 특정 비율로 섞어 학습한다. 이를 통해 기존의 영어 추론 능력을 잃지 않으면서 새로운 언어 체계를 습득하게 된다.

### 2. 토크나이저 효율화와 임베딩 확장
기존 Llama-3.1의 토크나이저 성능을 유지하면서 한국어 표현력을 높이기 위해 어휘 사전(Vocabulary)을 최적화한다. 영어 토큰 정보는 건드리지 않고 한국어 토큰의 밀도를 높여 두 언어 간의 연결 고리를 강화하는 방식이다.

### 3. 지식 전이(Knowledge Transfer)
영어 데이터로 학습된 모델의 강력한 논리적 사고 능력을 한국어로 전이시키는 과정을 거친다.

- 추론 능력 유지: 수학이나 코딩 같은 논리적 작업은 영어 데이터에서 배운 구조를 그대로 활용한다.
- 언어 정렬: SFT(지도 미세 조정) 단계에서 동일한 질문을 한국어와 영어로 번갈아 학습시켜, 언어에 상관없이 일관된 답변을 내놓도록 유도한다.

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = 'NCSOFT/Llama-VARCO-8B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model =AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

# llama-3 chat template 변환
Llama3 모델은 특정 chat template 형식으로 학습되어, 그 형식을 사용해야 최적 성능을 낼 수 있다. Chat template을 사용하지 않으면 모델이 대화 구조를 제대로 인식하지 못할 수 있다. open_ai 형식의 데이터를 llama-3 형식으로 변환한다.

LLaMA-3 채팅 포맷

LLaMA-3 채팅 포맷은 LLaMA-3 계열 챗봇 모델이 대화 내용을 이해하고 답변할 수 있도록 만들어진 입력 데이터 구조이다. 여러 역할(시스템, 유저, 어시스턴트)의 메시지를 특별한 토큰과 구조로 묶어서 하나의 프롬프트로 합치는 방식이다. 구조 예시는 아래와 같이 대화 흐름을 명확히 구분하는 토큰들이 사용된다.

<|begin_of_text|>

<|start_header_id|>system<|end_header_id|>
[시스템 역할 지침]<|eot_id|>

<|start_header_id|>user<|end_header_id|>
[유저 질문]<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
[모델의 답변]<|eot_id|>

- <|begin_of_text|>: 전체 프롬프트의 시작을 알리는 토큰
- <|start_header_id|>role<|end_header_id|>: 각 메시지의 역할 구분(시스템, 유저, 어시스턴트 등)
- 각 메시지 끝에 <|eot_id|>: 하나의 메시지 블록이 끝났음을 알림
- 마지막 assistant 블록은 응답 생성 위치를 가리킨다.

왜 이 포맷이 필요할까?

- 모델이 “어디까지가 시스템 안내, 어디서부터가 유저 질문, 어디서부터가 답변인지” 정확하게 파악할 수 있다.
- 여러 턴(turn)의 대화가 이어질 때도 메시지 경계를 명확히 구분해 혼동 없이 맥락을 유지할 수 있다.
- LLaMA-3 계열 모델은 이런 포맷으로 학습되어 있기 때문에 실전 파인튜닝/추론 시에도 반드시 이 구조로 입력해야 기대하는 챗봇 성능을 발휘할 수 있다.

In [12]:
# apply_chat_template 함수
# openai 방식의 메세지를 llama3 방식으로 변환
text = tokenizer.apply_chat_template(train_dataset[128]['messages'],tokenize=False)
print(text)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,
이유/근거 등을 분석하는 금융 분석 전문가입니다.

다음 출력 지시사항을 지켜주세요.
1. 뉴스와 종목간의 영향성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종복간의 영향성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정 영향이 예상되는 종목이 있다면, negaitive_stocks, negaitive_keywords, negaitive_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])를 작성하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

에어부산 울란바토르·오사카 노선 재개
에어부산이 김해국제공항에서 출발하는 몽골 울란바토르와 일본 오사카 노선 운항을 각각 주 2회 일정으로 코로나19 팬데믹 사태 이후 28개월 만에 재개한다고 1일 밝혔다. 부산 울란바토르 노선은 김해국제공항에서 오전 8시 25분에 출발해 현지 공항에 오전 11시 40분 도착하고 귀국편은 오후 1시에 출발해 김해공항에 오후 5시 30분 도착하는 일정으로 주 2회 운항한다. 몽골은 입국 시 코로나19 검사와 백신 접종 여부를 확인하지 않아 자유롭게 여행이 가능한 국가다. 부산 오사카 노선은 김해공항에서 오전 8시 35분에 출발해 간사이공항에 오전 10시 도착 귀국편은 간사이공항에서 낮 12시에 출발해 김해공항에 오후 1시 30분 도착하는

## data_collator 함수

data_collator 함수는 학습 과정에서 여러 개의 샘플을 하나의 미니배치(batch)로 묶고, 모델이 바로 학습할 수 있는 형태로 변환하는 역할을 한다.

- 미니배치(batch) 데이터를 모델이 바로 학습할 수 있는 형태(토큰, 마스크, 정답)로 변환한다.
- 특히 아래와 같은 LLaMA-3 채팅 포맷을 사용할 때, “어디까지가 질문이고 어디서부터가 답변(assistant)인지”를 정확히 구분해서 모델이 정답(답변 부분)만 학습하도록 레이블을 지정한다.

### 1. 프롬프트 생성 (Prompt Construction)

입력받은 batch 데이터는 리스트 내에 여러 메시지(system, user, assistant)를 포함하는 사전(dict) 구조이다.

- Llama 3의 특수 토큰(`<|begin_of_text|>`, `<|start_header_id|>`, `<|eot_id|>`)을 사용하여 모든 대화 내용을 하나의 긴 문자열로 병합한다.
- 각 역할(role)의 시작과 끝을 명확히 구분하여 모델이 대화 맥락을 이해할 수 있도록 구성한다.

### 2. 토크나이즈 및 패딩 (Tokenization)

병합된 문자열 리스트를 `tokenizer`를 통해 숫자 ID(`input_ids`)로 변환한다.

- `padding=True`: 배치 내의 문장들 중 가장 긴 문장을 기준으로 길이를 맞춘다.
- `truncation=True`: `max_length`를 초과하는 데이터는 절단한다.
- `return_tensors="pt"`: PyTorch 텐서 형식으로 결과를 반환한다.

### 3. 레이블 생성 및 Loss Masking

모델이 사용자의 질문이 아닌 모델의 답변(assistant) 부분에 대해서만 학습하도록 설정한다.

- `-100` 값의 의미: PyTorch의 `CrossEntropyLoss`는 레이블 값이 `-100`인 경우 손실(Loss) 계산에서 제외한다. 이를 통해 모델은 질문 부분을 예측하려고 노력하지 않고, 답변 부분의 정확도에만 집중하게 된다.
- 구간 탐색: `assistant_tokens`를 기준으로 답변이 시작되는 위치를 찾고, `<|eot_id|>` 토큰이 나오는 지점까지의 인덱스를 추출한다.
- 값 복사: 해당 구간의 `labels`에만 실제 `input_ids` 값을 복사하여 넣는다.

In [13]:
def data_collator(batch, tokenizer=tokenizer, max_length=8192):
    # 1. 프롬프트 생성
    prompts = []
    for example in batch:
        prompt = '<|begin_of_text|>'
        for msg in example['messages']:
            role = msg['role']
            content = msg['content'].strip()
            prompt += f"<|start_header_id|>{role}<|end_header_id|>\n{content}<|eot_id|>"
        prompts.append(prompt)
    # display(prompts)

    # 2. 토큰처리/패딩/텐서변환
    tokenized = tokenizer(
        prompts,
        truncation=True,
        max_length=max_length,
        padding=True, # 배치 내에서 가장 긴 텍스트 기준 패딩 처리
        return_tensors='pt'
    )
    input_ids = tokenized['input_ids']
    attention_mask = tokenized['attention_mask']
    # print(tokenized)
    # print(len(tokenized['input_ids'][0]))
    # print(len((tokenized['input_ids'][1])))
    # print(len((tokenized['attention_mask'][0])))
    # print(len((tokenized['attention_mask'][1])))

    # 3. 라벨 생성
    labels = torch.full_like(input_ids, fill_value=-100)
    # print(labels.shape)

    assistant_header = '<|start_header_id|>assistant<|end_header_id|>\n'
    assistant_token_id = tokenizer.encode(assistant_header, add_special_tokens=False)
    # print(assistant_token_id)
    eot_token = '<|eot_id|>'
    eot_token_id = tokenizer.encode(eot_token, add_special_tokens=False)
    # print(eot_token_id)

    for i, ids in enumerate(input_ids):
        ids_list = ids.tolist()

        # assistant 답변 위치 찾기
        start = None
        for idx in range(len(ids_list) - len(assistant_token_id) + 1):
            if ids_list[idx: idx + len(assistant_token_id)] == assistant_token_id:
                start = idx + len(assistant_token_id)
                break

        # 답변 끝 위치 찾기
        if start is not None:
            end = None
            for idx in range(start, len(ids_list) - len(eot_token_id) + 1):
                if  ids_list[idx: idx + len(eot_token_id)] == eot_token_id:
                    end = idx + len(eot_token_id)
                    break
        
        # print(f'{i}: {start} ~ {end}')

        # 정답 부분은 -100 이 아닌 실제 값으로 변환
        labels[i, start:end] = input_ids[i, start:end]

    return {
        'input_ids' : input_ids,
        'attention_mask' : attention_mask,
        'labels' : labels
    }

data_collator([train_dataset[0], train_dataset[1]])

{'input_ids': tensor([[128000, 128006,   9125,  ...,   1210,     92, 128009],
         [128000, 128006,   9125,  ...,      0,      0,      0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0]]),
 'labels': tensor([[  -100,   -100,   -100,  ...,   1210,     92, 128009],
         [  -100,   -100,   -100,  ...,   -100,   -100,   -100]])}

## Causal Language Model 파인튜닝: input_ids와 labels 구조 이해

### 데이터 구조

Causal Language Model을 파인튜닝할 때 학습 데이터는 보통 `input_ids`와 `labels`로 구성된다.

~~~text
input_ids: [system_tokens..., user_tokens..., assistant_tokens...]   # 프롬프트 + 정답 전체 시퀀스
labels:    [-100, -100, ..., -100, assistant_tokens...]              # assistant 답변 구간만 학습 대상
~~~

| 항목 | 내용 |
|---|---|
| Input IDs | 프롬프트 + 정답 전체 시퀀스 |
| Labels | `-100`은 프롬프트 구간, 실제 토큰 값은 답변 구간 |
| 결과 | 모델은 입력을 다 보지만, 오직 답변 부분을 예측하는 과정에서만 학습이 일어남 |

### 질문에 해당하는 input_ids에 이미 답이 포함되어 있다?

처음 보면 이런 의문이 생길 수 있다.

"답이 이미 있는데 어떻게 학습하는가?"

모델은 정답을 "보면서" 각 위치에서 올바른 다음 토큰을 예측하는 법을 배운다. 마치 학생이 모범답안을 보며 "이 상황에서는 이렇게 답해야 한다"를 학습하는 것과 같다. 이것이 현대 LLM 파인튜닝의 핵심 메커니즘이다.

### 1. 인과적 언어 모델링(Causal Language Modeling)

LLM(Llama, GPT 등)은 이전 토큰들을 보고 다음 토큰을 예측하는 방식으로 학습한다. 따라서 학습 데이터에는 프롬프트와 정답이 모두 포함된 전체 문장이 들어가야 한다.

- 학습 원리: 모델은 n번째 토큰까지를 입력으로 받아 n+1번째 토큰을 예측한다.
- 구조: `input_ids`가 `[A, B, C, D]`라면, 모델은 내부적으로 `A`를 보고 `B`를, `A, B`를 보고 `C`를 예측하는 과정을 동시에 수행한다.

### 2. Teacher Forcing 기법

학습 시에는 모델이 이전 단계에서 직접 생성한 토큰을 다음 입력으로 사용하는 것이 아니라, 정답 시퀀스에 있는 실제 이전 토큰을 입력으로 사용한다. 이를 Teacher Forcing이라고 한다.

예를 들어 다음과 같은 시퀀스가 있다고 하자.

~~~text
Position:  [0, 1, 2, 3, 4, 5, 6, 7, 8]
input_ids: [A, B, C, D, E, F, G, H, I]
labels:    [-100, -100, -100, -100, E, F, G, H, I]
~~~

이때 학습 과정은 다음과 같이 이해할 수 있다.

- Position 4: A, B, C, D를 보고 → E 예측
- Position 5: A, B, C, D, E를 보고 → F 예측
- Position 6: A, B, C, D, E, F를 보고 → G 예측

즉, `input_ids`에는 전체 토큰이 들어 있지만, 모델이 특정 위치의 토큰을 예측할 때는 그 위치 이전의 토큰만 참고한다. 뒤쪽 정답 토큰을 미리 보고 맞히는 구조가 아니다.

## 3. Labels와 Loss 계산의 역할

`input_ids`에 정답이 포함되어 있더라도, 모델이 모든 구간에 대해 학습(손실 계산)을 수행하는 것은 아니다. 이때 중요한 역할을 하는 것이 코드에 작성된 `labels`이다.

- `-100`의 의미: PyTorch의 `CrossEntropyLoss`는 기본적으로 레이블 값이 `-100`인 위치를 무시(ignore)한다.
- 학습 차단: 코드에서 프롬프트(User 질문 등) 구간의 레이블을 `-100`으로 설정했기 때문에, 모델이 프롬프트 내용을 예측하며 발생하는 오차는 학습에 반영되지 않는다.
- 학습 집중: 오직 `assistant`의 답변 구간에 해당하는 `labels`만 실제 `input_ids` 값을 가지므로, 모델은 "프롬프트가 주어졌을 때 정답을 생성하는 방법"에 대해서만 가중치를 업데이트한다.

## 학습 vs 추론의 차이

### 학습 시

~~~text
input_ids: <system>당신은 금융분석가</system><user>뉴스내용</user><assistant>분석결과</assistant>
labels:    [-100, -100, ..., -100, 분석결과_토큰들]
~~~

학습 시에는 프롬프트와 정답을 모두 넣는다.  
하지만 `labels`에서 프롬프트 구간은 `-100`으로 처리되기 때문에, 실제 손실 계산은 assistant 답변 구간에서만 일어난다.

### 추론 시

~~~text
input:  <system>당신은 금융분석가</system><user>뉴스내용</user><assistant>
output: 분석결과
~~~

추론 시에는 정답을 넣지 않는다.  
모델은 프롬프트만 입력받고, `<assistant>` 이후부터 분석 결과를 한 토큰씩 생성한다.

### 핵심 정리

- `input_ids`에는 프롬프트와 정답이 모두 들어간다.
- `labels`는 어느 부분을 학습할지 지정하는 역할을 한다.
- `labels`가 `-100`인 구간은 Loss 계산에서 제외된다.
- 따라서 모델은 system/user 프롬프트를 외우는 것이 아니라, assistant 답변 구간을 생성하는 법을 학습한다.
- 학습 시에는 정답을 함께 넣지만, 추론 시에는 정답 없이 프롬프트만 넣고 모델이 답변을 생성한다.

In [14]:
# 변환된 결과 확인
example = train_dataset[128]
batch = data_collator([example])
print(f'{batch['input_ids'].shape}')
print(f'{batch['attention_mask'].shape}')
print(f'{batch['labels'].shape}')

torch.Size([1, 885])
torch.Size([1, 885])
torch.Size([1, 885])


## LORA

In [15]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                    # 저랭크행렬의 차원수
    lora_alpha=32,          # lora 가중치 적용 정도(alpha/r) 1배수 ~ 4배수
    lora_dropout=0.1,
    bias='none',
    # q_proj, k_proj, v_proj, o_proj, up_porj, down_proj, gate_proj -> transpomer attention 종류
    target_modules=['q_proj', 'v_proj'],    
    task_type='CAUSAL_LM' # text_generation
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


In [16]:
from trl import SFTConfig

hub_model_id = 'sim2084/Llama-VARCO-8b-news2stock-analyser'

sft_config = SFTConfig(
    output_dir="Llama-VARCO-8b-news2stock-analyzer",  # 학습 결과물과 체크포인트가 저장될 폴더명
    num_train_epochs=3,                              # 전체 학습 데이터를 몇 번 반복해서 학습할지 설정
    per_device_train_batch_size=2,                   # GPU 1개당 한 번에 처리할 배치 크기
    gradient_accumulation_steps=2,                   # 그래디언트를 2번 누적 후 한 번 업데이트, 실질 batch_size=2*2 효과
    gradient_checkpointing=True,                     # 중간 활성화값 저장을 줄여 VRAM 사용량 절약
    optim="adamw_torch_fused",                       # PyTorch fused AdamW 옵티마이저 사용, 속도와 효율 개선
    logging_steps=10,                                # 10 step마다 학습 로그 출력
    save_strategy="steps",                           # 일정 step마다 모델 체크포인트 저장
    save_steps=50,                                   # 50 step마다 체크포인트 저장
    bf16=True,                                       # bfloat16 정밀도 사용, VRAM 절약 및 학습 속도 개선
    learning_rate=1e-4,                              # 학습률 설정
    max_grad_norm=0.3,                               # 그래디언트 클리핑 기준값, 급격한 업데이트 방지
    warmup_ratio=0.03,                               # 전체 학습 초반 3% 구간 동안 학습률을 서서히 증가
    lr_scheduler_type="constant",                    # warmup 이후 학습률을 일정하게 유지
    push_to_hub=True,                                # 학습 완료 후 Hugging Face Hub에 업로드
    hub_model_id=hub_model_id,                       # Hub에 업로드할 모델 저장소 이름
    hub_token=True,                                  # 로그인된 Hugging Face 토큰 사용
    remove_unused_columns=False,                     # 데이터셋의 사용하지 않는 컬럼도 제거하지 않음
    dataset_kwargs={"skip_prepare_dataset": True},   # TRL의 기본 데이터 전처리를 건너뛰고 직접 만든 데이터 구조 사용
    report_to=[],                                    # wandb, tensorboard 등 외부 로깅 도구 사용 안 함
    label_names=["labels"],                          # 학습에 사용할 정답 레이블 컬럼 이름 지정
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    data_collator=data_collator
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 0}.


Step,Training Loss
10,1.533300
20,1.216320
30,1.133837
40,1.057401
50,1.055393
60,1.134213
70,1.105027
80,1.074967
90,0.977876
100,1.088575


TrainOutput(global_step=600, training_loss=0.974608473777771, metrics={'train_runtime': 1176.4972, 'train_samples_per_second': 2.04, 'train_steps_per_second': 0.51, 'total_flos': 1.598476545955922e+17, 'train_loss': 0.974608473777771})

## 평가

In [18]:
# 평가(테스트)용 프롬프트(입력)와 정답(라벨) 리스트를 각각 만듦
prompt_list = []
label_list = []

for messages in test_dataset["messages"]:                                                            # 테스트 데이터셋의 각 샘플에서 messages(대화 내용 리스트)를 하나씩 꺼냄.
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)      # messages 리스트를 LLaMA-3 채팅 포맷 텍스트로 변환
    input = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[0] + \
        '<|start_header_id|>assistant<|end_header_id|>\n'                                            # assistant(정답) 답변이 시작되기 전까지의 텍스트를 프롬프트로 만듦
    label = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[1].split('<|eot_id|>')[0]  # assistant 구간의 시작 이후부터 <|eot_id|>(답변 끝) 이전까지의 텍스트만 추출해 정답으로 만듦
    prompt_list.append(input)                                                                         # 입력(프롬프트) 텍스트를 리스트에 저장
    label_list.append(label)

In [19]:
prompt_list[100]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,\n이유/근거 등을 분석하는 금융 분석 전문가입니다.\n\n다음 출력 지시사항을 지켜주세요.\n1. 뉴스와 종목간의 영향성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종복간의 영향성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정 영향이 예상되는 종목이 있다면, negaitive_stocks, negaitive_keywords, negaitive_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])를 작성하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n공유수면 사용하려면 어입인 의견 들어야\n해수부 5일 개정 공유수면 관리 및 매립에 관한 법률 시행 헤럴드경제 홍태화 기자 앞으로 공유수면관리청이 어업·환경 등에 영향을 미칠 것으로 예상되는 공유수면 점용·사용 허가를 할 때 미리 어업인 등 이해관계자들의 의견을 들어야 한다. 해양수산부는 5일 이같은 내용이 담긴 개정 공유수면 관리 및 매립에 관한 법률과 같은 법 시행령·시행규칙이 이날부터 시행된다고 밝혔다. 바다·바닷가·하천 등 공유수면은 공유재이기 때문에 이를 점용·사용하기 위해서는 별도의 허가를 받아야 한다. 최근 해상풍력 발전시설 해변을 이용한 관광시설 등 대규모 시설이 공유수면을 장기적으로 점용·사용하는 경우가 늘어났지만 이해 관

In [20]:
label_list[100]

'\n{"stock_related":true,"summary":"해양수산부가 공유수면 점용·사용 허가 시 어업인 등 이해관계자 의견 수렴을 의무화하는 법률 개정 및 시행령을 6월 5일부터 시행한다. 공유수면(바다, 해변, 하천 등) 내 대규모 시설(해상풍력, 관광시설 등) 설치에 앞서 관련 업계, 어업인 등의 의견을 청취해야 하며, 피해 우려가 있을 경우 별도 의견 조사도 필요하다. 이는 사회적 갈등 예방과 피해 최소화가 목적이다.","positive_stocks":["동국산업","씨에스윈드","코오롱글로벌"],"positive_keywords":["해상풍력","관광시설","사회적 갈등 예방"],"positive_reasons":"어업인 등 이해관계자 의견 수렴이 체계화됨으로써 허가 절차의 예측 가능성이 높아지고, 사회적 갈등 최소화를 위한 로드맵이 명확해져 중장기적으로 해상풍력 및 관광시설 추진 기업들의 투자 리스크가 완화될 수 있음.","negative_stocks":["해성산업","사조산업","동원수산"],"negative_keywords":["어업인 피해","수산자원 보호","허가 심의 강화"],"negative_reasons":"어업인 등 기존 이용자들의 의견이 반영되며 신규 해상·관광시설 진출이 까다로워질 수 있고, 수산업 관련 기업에는 사업 제한이나 경쟁 심화 등 부정적 영향이 발생할 수 있음."}'

## 추론 모델 로드

In [21]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline
import torch

peft_model_id = hub_model_id # lora로 학습된 모델 저장소(adapter파일만 있음)

finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)

pipe = pipeline(
    'text-generation',
    model=finetuned_model,
    tokenizer=tokenizer
)

adapter_config.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/497 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/348 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

In [22]:
eos_token_id = tokenizer.encode('<|eot_id|>', add_special_tokens=False)[0]
eos_token_id

128009

In [23]:
# 평가셋을 위한 추론 함수
def test_inference(pipe, prompt):
    outputs = pipe(
        prompt,
        max_new_tokens=1024, # 출력토큰에 대한 제한 (max_length: 입출력 전체)
        eos_token_id=eos_token_id,
        do_sample=False # False: Greedy방식 작동(확률이 가장 토큰 선택. 일관된 답변)
    )
    assistant_start = len(prompt)
    return outputs[0]['generated_text'][assistant_start:].strip()

In [24]:
# 3건만 샘플링해서 추론
start = 12
end = 15
for prompt, label in zip(prompt_list[start:end], label_list[start:end]):
    print(f'prompt: \n{prompt}')
    print(f'label: \n{label}')
    print(f'response: \n{test_inference(pipe, prompt)}')
    print('-' * 100)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


prompt: 
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,
이유/근거 등을 분석하는 금융 분석 전문가입니다.

다음 출력 지시사항을 지켜주세요.
1. 뉴스와 종목간의 영향성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종복간의 영향성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정 영향이 예상되는 종목이 있다면, negaitive_stocks, negaitive_keywords, negaitive_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])를 작성하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

삼성바이오로직스 MSD와 2768억원 위탁생산계약 체결
미국 제약기업과 의약품 위탁생산 공급계약 지난해 9월 공시된 본 계약 체결 삼성바이오로직스 3공장 전경. ⓒ삼성바이오로직스 데일리안 이홍석 기자 삼성바이오로직스는 4일 공시를 통해 미국 제약기업 MSD MSD International Business GmbH 와 2768억2938만원 규모의 의약품 위탁생산 공급계약을 체결했다고 밝혔다. 이번 계약은 최근 매출액 대비 17.65% 규모로 계약기간은 2022년 7월 1일부터 2028년 12월 31일이다. 상기 계약금액은 고객사의 수요증가에 따라 3억8186만 달러로 증가할 수 있다. 이번 계약건은 지난해 9월 29일 공시된 ‘투자판단 관련 주요경영사항’

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


response: 
{"stock_related":true,"summary":"삼성바이오로직스는 미국 제약기업 MSD와 2768억2938만원 규모의 의약품 위탁생산 공급계약을 체결했다. 이번 계약은 2022년 7월부터 2028년 12월까지 지속되며, 계약금액은 수요 증가에 따라 추가로 증가할 수 있다. 이는 지난해 9월에 의향서를 체결했을 때 491억원으로 공시했던 금액보다 크게 확대된 규모다.","positive_stocks":["삼성바이오로직스"],"positive_keywords":["MSD와 위탁생산계약","대규모 계약","수요 증가 가능성","수익성 개선"],"positive_reasons":"MSD와 대규모 위탁생산 계약을 체결함으로써 삼성바이오로직스의 매출과 수익성이 크게 개선될 전망이다. 수요 증가에 따라 추가 수익도 기대된다.","negative_stocks":[],"negative_keywords":[],"negative_reasons":""}
----------------------------------------------------------------------------------------------------
prompt: 
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,
이유/근거 등을 분석하는 금융 분석 전문가입니다.

다음 출력 지시사항을 지켜주세요.
1. 뉴스와 종목간의 영향성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종복간의 영향성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


response: 
{"stock_related":true,"summary":"LG전자와 SM엔터테인먼트가 디지털 피트니스 콘텐츠 브랜드 '피트니스 캔디'를 출시했습니다. 이 브랜드는 MZ세대를 겨냥한 모바일 기반의 맞춤형 운동 데이터 플랫폼, K-POP 콘텐츠, 스마트 가전 기술 등이 결합된 서비스로, 피트니스 시장의 디지털화와 트렌드 변화에 대응하는 전략입니다. LG전자는 스마트 가전 패러다임으로의 전환, SM엔터는 K-POP 콘텐츠와 글로벌 피트니스 시장 진출을 강조했습니다.","positive_stocks":["LG전자","SM엔터테인먼트"],"positive_keywords":["디지털 피트니스","스마트 가전","K-POP 콘텐츠","MZ세대","홈트레이닝","데이터 플랫폼"],"positive_reasons":"LG전자는 스마트 가전 및 디지털 라이프스타일 시장에서 기술력과 데이터 활용 경험을 바탕으로 성장세를 이어갈 수 있으며, SM엔터는 K-POP 콘텐츠를 활용한 글로벌 피트니스 시장 진출에 성공할 수 있습니다. 두 회사의 협업은 새로운 수익원 창출과 브랜드 가치 제고에 기여할 것으로 기대됩니다.","negative_stocks":[],"negative_keywords":[],"negative_reasons":""}
----------------------------------------------------------------------------------------------------
prompt: 
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,
이유/근거 등을 분석하는 금융 분석 전문가입니다.

다음 출력 지시사항을 지켜주세요.
1. 뉴스와 종목간의 영향성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스

## 뉴스 원문 추론

In [25]:
def inference(news):
    messages = [
        {'role': 'system', 'content': """
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 
이유/근거 등을 분석하는 금융분석 전문가입니다.

다음 출력 지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
"""},
        {'role': 'user', 'content': news}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    
    outputs = pipe(
        prompt,
        max_new_tokens=1024, # 출력토큰에 대한 제한 (max_length: 입출력 전체)
        eos_token_id=eos_token_id,
        do_sample=False, # False: Greedy방식 작동(확률이 가장 토큰 선택. 일관된 답변)
    )
    assistant_start = len(prompt)
    return outputs[0]['generated_text'][assistant_start:].strip()

In [26]:
inference(news="""
(뉴욕=연합뉴스) 진정호 연합인포맥스 특파원 = 뉴욕증시의 3대 주가지수가 지루한 흐름을 보인 끝에 보합권에서 혼조로 마감했다.

도널드 트럼프 미국 대통령이 관세 부과 시점을 8월 1일 이후로는 연장하지 않겠다고 공언했으나 그가 숱하게 말을 번복해왔던 만큼 시장은 크게 개의치 않았다.

트럼프는 또 구리에 50%의 관세를 부과하겠다고 밝혔으나 이 또한 예상된 재료였던 만큼 투심을 흔들지는 못했다.

뉴욕증권거래소
[연합뉴스 자료사진]
뉴욕증권거래소
[연합뉴스 자료사진]


8일(미국 동부시간) 뉴욕증권거래소(NYSE)에서 다우존스30산업평균지수는 전장보다 165.60포인트(0.37%) 내린 44,240.76에 거래를 마감했다.

스탠더드앤드푸어스(S&P)500지수는 전장보다 4.46포인트(0.07%) 떨어진 6,225.52, 나스닥종합지수는 5.95포인트(0.03%) 오른 20,418.46에 장을 마쳤다.

트럼프는 이날도 관세 관련 발언을 쏟아냈으나 증시도 내성이 생긴 듯 보합권에서 한산한 움직임을 보였다.

트럼프는 자신의 소셜미디어 트루스소셜에 게시한 글에서 "관세는 2025년 8월 1일부터 부과되기 시작할 것"이라며 "(기한) 연장은 허용되지 않을 것"이라고 말했다.

이는 전날 트럼프가 내놓은 발언과 배치되는 것이다. 트럼프는 전날 한국과 일본 등 14개국에 관세 서한을 보내는 한편 관세 부과 시점을 8월 1일로 연기했으나 협상 상대방이 좋은 제안을 가져오면 관세 부과 시점이 더 미뤄질 수 있다고 말한 바 있다.

트럼프는 또 이르면 이달 말 반도체와 의약품 등 주요 품목에 대해 관세를 부과할 계획이라는 점을 알렸다. 반도체에 대해선 구체적인 관세율과 부과 시점 등이 발표되지 않았으나 의약품은 최대 200%의 관세가 부과될 수 있다고 그는 말했다.

뱅크오브아메리카의 안토니오 가브리엘 이코노미스트는 "전날 발표된 관세가 확정된 것은 아니라고 본다"면서도 "관세가 시행된다면 물가상승률은 약 0.1%포인트 상승하고 성장률은 비슷한 수준으로 하락할 것"이라고 말했다.

트럼프가 구리에 50%의 관세를 부과하기로 한 점은 장기적으로 인플레이션을 자극할 수 있다는 관측도 나온다.

구리는 제조업 전반에 소요되는 필수 요소인 만큼 관세발 인플레이션에도 취약할 수밖에 없다. 트럼프의 발표 이후 금속선물거래소 코멕스(COMEX)에서 구리선물 가격은 한때 17% 폭등하며 역대 최고치를 경신했다.

리베르타스웰스매니지먼트의 아담 쿠스 대표는 "우리는 미국 산업을 보호하기 위해 정책을 무기화하는 움직임을 보고 있지만 이는 인플레이션 공포를 부채질할 것"이라며 "관세 위협이 공식 정책이 되면 힘을 발휘할 수 있겠지만 대부분의 정치 랠리가 그렇듯 짧은 도화선이 될 가능성이 크다"고 말했다.

업종별로는 에너지가 2.72% 급등했고 유틸리티와 필수소비재는 1% 이상 하락했다.

시가총액 1조달러 이상의 거대 기술기업 중에선 엔비디아와 테슬라가 1% 이상 상승했다.

엔비디아는 이날 강세로 시총이 3조9천억달러를 넘어서며 사상 최초 4조달러를 눈앞에 두게 됐다.

엔비디아에 대한 기대감이 반도체 업계 전반으로 퍼지면서 필라델피아 반도체지수도 1.80% 뛰었다. 해당 지수를 구성하는 30개 종목 중 27개가 강세였다.

트럼프가 친환경 에너지 보조금 축소를 골자로 한 행정명령에 서명했다는 소식에 에너지 관련주가 급등했다.

셰브런은 3.96%, 엑손 모빌은 2.77% 상승했다.

반면 태양광 관련주들은 일제히 약세였다. 선런의 주가는 전일 대비 11%, 퍼스트 솔라는 6% 넘게 떨어졌다.

은행주들 역시 이날 약세였다. 은행권의 2분기 실적 시즌을 앞두고 HSBC가 대형 은행에 대한 투자의견을 하향 조정한 여파다.

JP모건체이스와 뱅크오브아메리카의 주가는 3% 넘게 떨어졌고 모건스탠리와 골드만삭스도 2% 가까이 하락했다.

시카고상품거래소(CME) 페드워치툴에 따르면 연방기금금리 선물시장은 7월 기준금리 동결 확률을 95.3%로 유지했다. 연말까지 2회 금리 인하될 확률은 43.7%로 반영되며 가장 가능성이 높게 점쳐졌다.

시카고옵션거래소(CBOE) 변동성 지수(VIX)는 0.98포인트(5.51%) 떨어진 16.81을 기록했다.
""")

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'{"stock_related":true,"summary":"뉴욕증시의 주요 3대 주가지수(다우존스, S&P500, 나스닥)가 보합권에서 혼조로 마감했다. 트럼프 대통령의 관세 관련 발언에도 불구하고 시장은 크게 반응하지 않았다. 반도체와 의약품 등 주요 품목에 관세 부과가 예고됐으나, 이는 예상된 정책이어서 투자심을 크게 흔들지 못했다. 에너지와 유틸리티, 필수소비재는 각각 강세와 하락을 보였고, 엔비디아와 테슬라 등 거대 기술주가 강세를 보였다. 반면 태양광 관련주와 은행주들은 약세를 기록했다.","positive_stocks":["엔비디아","테슬라","셰브런","엑손 모빌"],"positive_keywords":["반도체","에너지","기술주","관세 부과 미확정"],"positive_reasons":"트럼프의 관세 부과 시점 연기와 반도체, 에너지 관련주에 대한 기대감, 그리고 기술주(엔비디아, 테슬라)의 강세가 시장에 긍정적 영향을 미쳤다.","negative_stocks":["태양광 관련주","은행주(HSBC, JP모건체이스, 뱅크오브아메리카, 모건스탠리, 골드만삭스)"],"negative_keywords":["태양광","은행주","투자의견 하향조정"],"negative_reasons":"태양광 관련주와 대형 은행주가 각각 약세를 기록했으며, HSBC의 투자의견 하향조정 등으로 인해 금융주 전반에 부정적 영향을 미쳤다."}'

## Basemodel 비교

In [ ]:
# LoRA 파인튜닝 전(Base) vs 파인튜닝 후(LoRA) 모델 답변 비교
from transformers import AutoModelForCausalLM, pipeline

base_model_id = "NCSOFT/Llama-VARCO-8B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

base_pipe = pipeline("text-generation", model=base_model, tokenizer=tokenizer)

for idx, (prompt, label) in enumerate(zip(prompt_list[12:15], label_list[12:15])):
    print(f"샘플 {idx + 1}")
    base_resp = test_inference(base_pipe, prompt)
    lora_resp = test_inference(pipe, prompt)
    print(f"  [Base - 파인튜닝 전]\n{base_resp}")
    print(f"  [LoRA - 파인튜닝 후]\n{lora_resp}")
    print(f"  [Label]\n{label}")
    print("-" * 50)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


샘플 1


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Base - 파인튜닝 전]
stock_related: True
summary: 삼성바이오로직스는 미국 제약기업 MSD와 2768억2938만원 규모의 의약품 위탁생산 공급계약을 체결했습니다. 이번 계약은 2022년 7월부터 2028년 12월까지 지속되며, 고객사 수요에 따라 계약금액이 증가할 수 있습니다. 이는 지난해 9월에 공시된 투자판단 관련 주요경영사항에 따른 계약 의향서의 실현입니다.

positive_stocks:
- 삼성바이오로직스

positive_keywords:
- 위탁생산 계약 체결
- 의약품 공급 계약
- 2768억2938만원 규모

positive_reasons:
- 이번 계약은 삼성바이오로직스의 생산 능력과 수요 증가에 따른 매출 확대를 기대할 수 있게 합니다.
- 계약 기간 동안의 안정적인 수익원으로 작용할 것으로 보입니다.
- 미국 제약기업과의 파트너십 강화로 글로벌 시장에서의 경쟁력 향상 기대. 

negaitive_stocks:
- 없음

negaitive_keywords:
- 없음

negaitive_reasons:
- 없음
  [LoRA - 파인튜닝 후]
{"stock_related":true,"summary":"삼성바이오로직스는 미국 제약기업 MSD와 2768억2938만원 규모의 의약품 위탁생산 공급계약을 체결했다. 이번 계약은 2022년 7월부터 2028년 12월까지 지속되며, 계약금액은 수요 증가에 따라 추가로 증가할 수 있다. 이는 지난해 9월에 의향서를 체결했을 때 491억원으로 공시했던 금액보다 크게 확대된 규모다.","positive_stocks":["삼성바이오로직스"],"positive_keywords":["MSD와 위탁생산계약","대규모 계약","수요 증가 가능성","수익성 개선"],"positive_reasons":"MSD와 대규모 위탁생산 계약을 체결함으로써 삼성바이오로직스의 매출과 수익성이 크게 개선될 전망이다. 수요 증가에 따라 추가 수익도 기대된다.","negative_stocks":[],"nega

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Base - 파인튜닝 전]
stock_related: False
summary: LG전자와 SM엔터테인먼트가 디지털 피트니스 콘텐츠 합작 브랜드 '피트니스 캔디'를 발표했습니다. '피트니스 캔디'는 MZ세대를 겨냥한 모바일 기반의 차세대 운동 데이터 플랫폼으로, K POP의 댄스와 리듬을 리마스터링한 디지털 라이프스타일 무브먼트를 제공할 예정입니다. 이 브랜드는 LG전자의 디지털 기술력과 고객 데이터 경영 경험, SM엔터테인먼트의 음악 콘텐츠를 바탕으로 건전하고 건강한 피트니스 습관을 재발견하는 디지털 피트니스 콘텐츠 프로젝트입니다.

positive_stocks: 
positive_keywords:
positive_reasons:
- LG전자의 디지털 기술력과 고객 데이터 경영 경험을 활용하여 개인 맞춤형 피트니스 서비스를 제공할 수 있음
- SM엔터테인먼트의 K POP 음악과 댄스 콘텐츠가 '피트니스 캔디'의 디지털 라이프스타일 무브먼트에 큰 도움을 줄 것으로 기대됨

negaitive_stocks:
negaitive_keywords:
negaitive_reasons:
- 현재 시장 상황에서 새로운 피트니스 플랫폼을 성공적으로 런칭하는 것은 매우 경쟁적일 수 있음
- 특히 코로나19 팬데믹 이후 온라인 피트니스 시장의 포화 상태가 우려됨
- '피트니스 캔디'가 단순히 트렌드에 맞춘 서비스가 아니라 지속 가능한 사업 모델을 구축하는 데 성공해야 할 것임
- K POP 콘텐츠의 효과가 전 세계적으로 일관되게 나타나기까지는 시간이 필요할 수 있음
- '피트니스 캔디'가 고객의 개인 데이터를 적절히 보호하고 윤리적으로 활용하는지에 대한 우려도 존재함
- 디지털 피트니스 시장의 성장률이 둔화되는 추세일 수 있음
- '피트니스 캔디'의 경쟁사로는 ZOOM, Nike Training Club, Fitbit Coach 등이 있음
- '피트니스 캔디'가 기존 플랫폼들과 차별화된 서비스를 제공하지 못한다면 시장 점유율 확보에 어려움을 겪을 수 있음
- '피트니스 캔디'

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
